# Customer Master Data Migration — Transformation Layer

## Objective
Build a traceable transformation layer for the legacy customer extract and produce `CustomerLoad.csv` for the target SAP entity.

The notebook:
1. Loads the supplied `CustomerExtract.csv`.
2. Loads the `CustomerSpec` worksheet from `CustomerSpec.xlsx`.
3. Profiles the source data.
4. Performs visible data-quality checks.
5. Cleans `NAME1` special characters.
6. Applies the transformation rules in the specification.
7. Validates the transformed file.
8. Writes `output/CustomerLoad.csv` and `output/data_quality_report.csv`.

> **Traceability note:** The source extract contains 100 records and 183 columns. The specification marks 50 target fields as `Field Used = Y`. The target file therefore contains those 50 SAP fields in specification order.


## Transformation decisions / assumptions

- `BUKRS` is populated with `G100`, `VKORG` with `G100`, `VTWEG` with `20`, and `SPART` with `10`, exactly as specified.
- Fields described as **Copy Existing** are copied from the legacy extract.
- `NAME1` is cleaned by removing punctuation/special characters and normalizing whitespace while retaining Unicode letters/numbers.
- `SPRAS` is set to `EN` because the specification states both systems use EN.
- `ERDAT` and `ERNAM` are left blank because they are **DEF by System**.
- `LIFNR` is left blank because the specification says the mapping is pending a business decision.
- New-system fields with no approved source mapping are left blank rather than passing through unexpected legacy data.
- `AUFSD` is copied because the specification explicitly says **Copy Existing**, even though it is not flagged as legacy-utilized; this conflict is surfaced by the DQ report.
- `KATR6`, `KATR10`, `KDKG1`, and `KDKG2` contain source data but are not flagged as legacy-utilized and have no approved copy rule, so they are not loaded. They are surfaced as DQ warnings.


In [33]:
# Import the transformation functions.
from pathlib import Path
import pandas as pd
from transform import load_source, load_spec, profile_and_checks, build_output

BASE_DIR = Path.cwd()
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

df = load_source()
spec = load_spec()

print(f"Source shape: {df.shape}")
print(f"Specification shape: {spec.shape}")


Source shape: (100, 183)
Specification shape: (174, 13)


c:\Users\Sana Mulani\customer_data_migration_submission\.venv\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


## 1. Initial data profile

The following checks provide an auditable profile of row/column counts, populated fields, missing values and uniqueness for the customer key.

In [34]:
# Profile output for traceability.
profile = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(df[c].dtype) for c in df.columns],
    "non_blank": [int(df[c].astype(str).str.strip().ne("").sum()) for c in df.columns],
    "blank": [int(df[c].astype(str).str.strip().eq("").sum()) for c in df.columns],
    "unique": [int(df[c].nunique(dropna=False)) for c in df.columns],
})
display(profile.head(30))
print(f"Total rows: {len(df):,}")
print(f"Total columns: {len(df.columns):,}")
print(f"Unique KUNNR: {df['KUNNR'].nunique():,}")
print(f"Duplicate KUNNR records: {df['KUNNR'].duplicated(keep=False).sum():,}")


,column,dtype,non_blank,blank,unique
0,KUNNR,str,100,0,100
1,LAND1,str,100,0,1
2,NAME1,str,100,0,90
3,NAME2,str,0,100,1
4,ORT01,str,100,0,50
5,PSTLZ,str,100,0,92
6,REGIO,str,100,0,30
7,SORTL,str,100,0,27
8,STRAS,str,100,0,100
9,TELF1,str,80,20,25


Total rows: 100
Total columns: 183
Unique KUNNR: 100
Duplicate KUNNR records: 0


## 2. Specification coverage

The specification is used as the source of truth for target-field selection and transformation logic.

In [35]:
used = spec[spec["Field Used"].fillna("").astype(str).str.strip().str.upper().eq("Y")].copy()

coverage = used[[
    "SAP FIELD DESCRIPTION", "SAP FIELD",
    "Field Utilized in LEGACY System",
    "Field Utilized in NEW System",
    "LEGACY REQ", "NEW REQ", "FIT/GAP",
    "Selection/Transformation Logic"
]].copy()

print(f"Target fields marked Field Used = Y: {len(coverage)}")
display(coverage)


Target fields marked Field Used = Y: 50


,SAP FIELD DESCRIPTION,SAP FIELD,Field Utilized in LEGACY System,Field Utilized in NEW System,LEGACY REQ,NEW REQ,FIT/GAP,Selection/Transformation Logic
0,Account Group,KTOKD,Y,Y,REQ,REQ,GAP,If decision to have new customer account numbe...
1,Customer Account Number,KUNNR,Y,Y,REQ,REQ,GAP,If decision to have new customer account numbe...
2,Company Code,BUKRS,Y,Y,REQ,REQ,GAP,Use Company Code G100
3,Sales Organization,VKORG,Y,Y,REQ,REQ,GAP,Use Sales Organization G100
4,Distribution Channel,VTWEG,Y,Y,OPT,OPT,FIT,Use Existing Distribution Channel 20
5,Division,SPART,Y,Y,OPT,OPT,FIT,Use Existing Division 10
7,Country,LAND1,Y,Y,REQ PROC,REQ PROC,FIT,Copy Existing for Customer records coming from...
8,Name 1,NAME1,Y,Y,REQ PROC,REQ PROC,FIT,Copy Existing for Customer records coming from...
9,Name 2,NAME2,Y,Y,OPT,OPT,FIT,Copy Existing for Customer records coming from...
10,City,ORT01,Y,Y,REQ PROC,REQ PROC,FIT,Copy Existing for Customer records coming from...


## 3. Data-quality checks

Checks include:
- record count;
- duplicate/missing customer account numbers;
- required legacy-field completeness;
- populated source fields that are not flagged as legacy-utilized;
- target length compliance;
- special characters in `NAME1`;
- source language versus the target `EN` rule;
- unresolved vendor mapping;
- populated fields that are both `Field Used = Y` and not flagged as legacy-utilized.


In [36]:
# Visible DQ results for traceability.
dq = profile_and_checks(df, spec)
display(dq)

print("\nDQ status summary:")
display(dq["status"].value_counts().rename_axis("status").reset_index(name="checks"))


,check,status,count,details
0,Row count,PASS,100,Loaded 100 customer records.
1,Duplicate KUNNR,PASS,0,Customer Account Number must be unique.
2,Missing KUNNR,PASS,0,Customer Account Number is required.
3,Required source completeness: KTOKD,PASS,0,0 blank value(s) found.
4,Required source completeness: KUNNR,PASS,0,0 blank value(s) found.
5,Required target field source availability: BUKRS,PASS,0,Source field is not required because the speci...
6,Required target field source availability: VKORG,PASS,0,Source field is not required because the speci...
7,Required source completeness: LAND1,PASS,0,0 blank value(s) found.
8,Required source completeness: NAME1,PASS,0,0 blank value(s) found.
9,Required source completeness: ORT01,PASS,0,0 blank value(s) found.



DQ status summary:


,status,checks
0,PASS,19
1,WARN,5


In [37]:
# Show the unexpected populated source fields explicitly.
legacy_fields = set(
    spec.loc[
        spec["Field Utilized in LEGACY System"].fillna("").astype(str).str.strip().str.upper().eq("Y"),
        "SAP FIELD"
    ].dropna().astype(str).str.strip()
)

populated_unflagged = [
    c for c in df.columns
    if df[c].astype(str).str.strip().ne("").any() and c not in legacy_fields
]

unexpected_profile = pd.DataFrame({
    "field": populated_unflagged,
    "non_blank_records": [int(df[c].astype(str).str.strip().ne("").sum()) for c in populated_unflagged]
}).sort_values("non_blank_records", ascending=False)

display(unexpected_profile)


,field,non_blank_records
1,ADRNR,100
31,/VSO/R_ULD_SIDE,100
2,MCOD1,100
3,MCOD3,100
8,BBSNR,100
7,BBBNR,100
14,UMJAH,100
15,JMZAH,100
9,BUBKZ,100
16,JMJAH,100


## 4. Transform the data

In [38]:
# Apply the documented transformation rules.
customer_load = build_output(df, spec)

print(f"Transformed shape: {customer_load.shape}")
display(customer_load.head(10))


Transformed shape: (100, 50)


,KTOKD,KUNNR,BUKRS,VKORG,VTWEG,SPART,LAND1,NAME1,NAME2,ORT01,...,KATR8,KATR9,KATR10,TXJCD,KDKG1,KDKG2,KDKG3,KDKG4,KDKG5,STCD5
0,1,1,G100,G100,20,10,ZZ,ZZZ ZZZZZZZ ZZ,,ZZZZZZ,...,,,,,,,,,,
1,1,3,G100,G100,20,10,ZZ,ZZZZZZZZZ ZZZZZZ ZZ,,ZZZZZZZ,...,,,,,,,,,,
2,1,7,G100,G100,20,10,ZZ,ZZZZZZZZZZZ ZZZZZZ ZZ,,ZZZZZZZ ZZZ ZZZZZZ,...,,,,,,,,,,
3,1,30,G100,G100,20,10,ZZ,ZZZZZZ ZZZZZZZZ ZZ,,ZZZZZZZZZZ,...,,,,,,,,,,
4,1,31,G100,G100,20,10,ZZ,ZZZZ ZZ,,ZZZZZZZZ,...,,,,,,,,,,
5,1,39,G100,G100,20,10,ZZ,ZZZZZZZZZ ZZZZZ ZZ,,ZZZZZZ,...,,,,,,,,,,
6,1,45,G100,G100,20,10,ZZ,ZZZZZZZZZ ZZZZZZ ZZZZZZZZZZ,,ZZZZZZ ZZ ZZZZZZ,...,,,,,,,,,,
7,1,46,G100,G100,20,10,ZZ,ZZZZZZ ZZZZ ZZ,,ZZ ZZZZ,...,,,,,,,,,,
8,1,47,G100,G100,20,10,ZZ,ZZZZZZZZZ ZZZZZZZZZ ZZ,,ZZ ZZZZ,...,,,,,,,,,,
9,1,48,G100,G100,20,10,ZZ,ZZZZZZZZ ZZZZZZ ZZZZZZZ,,ZZZZZZZZZZZ,...,,,,,,,,,,


In [39]:
# Confirm the NAME1 cleanup.
name_check = pd.DataFrame({
    "original": df["NAME1"].head(10),
    "cleaned": customer_load["NAME1"].head(10)
})
display(name_check)

,original,cleaned
0,ZZZ ZZZZZZZ ZZ,ZZZ ZZZZZZZ ZZ
1,ZZZZZZZZZ ZZZZZZ ZZ,ZZZZZZZZZ ZZZZZZ ZZ
2,ZZZZZZZZZZZ ZZZZZZ ZZ,ZZZZZZZZZZZ ZZZZZZ ZZ
3,ZZZZZZ ZZZZZZZZ ZZ,ZZZZZZ ZZZZZZZZ ZZ
4,ZZZZ ZZ,ZZZZ ZZ
5,ZZZZZZZZZ ZZZZZ ZZ,ZZZZZZZZZ ZZZZZ ZZ
6,ZZZZZZZZZ ZZZZZZ ZZZZZZZZZZ,ZZZZZZZZZ ZZZZZZ ZZZZZZZZZZ
7,ZZZZZZ ZZZZ ZZ,ZZZZZZ ZZZZ ZZ
8,ZZZZZZZZZ ZZZZZZZZZ ZZ,ZZZZZZZZZ ZZZZZZZZZ ZZ
9,ZZZZZZZZ ZZZZZZ ZZZZZZZ,ZZZZZZZZ ZZZZZZ ZZZZZZZ


## 5. Output validation

In [40]:
# Validate target structure and key transformations.
target_fields = used["SAP FIELD"].dropna().astype(str).str.strip().tolist()

assert customer_load.shape[0] == df.shape[0]
assert customer_load.columns.tolist() == target_fields
assert customer_load["KUNNR"].astype(str).str.strip().ne("").all()
assert customer_load["KUNNR"].is_unique
assert (customer_load["BUKRS"] == "G100").all()
assert (customer_load["VKORG"] == "G100").all()
assert (customer_load["VTWEG"] == "20").all()
assert (customer_load["SPART"] == "10").all()
assert (customer_load["SPRAS"] == "EN").all()
assert customer_load["NAME1"].map(lambda x: not bool(__import__("re").search(r"[^A-Za-z0-9\s\u00C0-\u024F]", str(x)))).all()

print("All structural and transformation assertions passed.")


All structural and transformation assertions passed.


In [41]:
# Save the final deliverables.
output_path = OUTPUT_DIR / "CustomerLoad.csv"
dq_path = OUTPUT_DIR / "data_quality_report.csv"

customer_load.to_csv(output_path, index=False)
dq.to_csv(dq_path, index=False)

print(f"Created: {output_path}")
print(f"Created: {dq_path}")
print(f"CustomerLoad.csv rows: {len(customer_load):,}")
print(f"CustomerLoad.csv columns: {len(customer_load.columns):,}")


Created: c:\Users\Sana Mulani\customer_data_migration_submission\output\CustomerLoad.csv
Created: c:\Users\Sana Mulani\customer_data_migration_submission\output\data_quality_report.csv
CustomerLoad.csv rows: 100
CustomerLoad.csv columns: 50


## 6. Final result

The transformation produces a 100-row, 50-column flat file named `CustomerLoad.csv`.

The DQ report is retained separately so warnings and review items are traceable without contaminating the SAP load file.
